# Gpd.ipynb

Will transform each pose into gpd, reduce dimensionality with PCA and avg them across individuals to construct frame vector


In [ ]:
import numpy as np
from utils.gpd_utils import*
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from collections import defaultdict


#importing cleaned annotations
try:
    with open('cleanedAnnots.pkl', 'rb') as f:
        annots = pickle.load(f)
except:
    print("Error")

## Computing gpd frames

In [ ]:
#computing gpd frames

for day in annots:
    for operation in annots[day]:
        for frame in annots[day][operation]:
            for pose in frame['poses']:
                pose['gpdRep'] = computeGpd(pose['keypoints'])



In [ ]:
#checkign results and shape
exKp = annots['day1']['operation_1'][0]['poses'][0]['gpdRep']
print(exKp)
print(np.shape(exKp))

In [ ]:
#stacking all poses into matrix
gpdList = [] 

frameIds = []  #will be used later when averaging gpd vectors per frame

for day in annots:
    for operation in annots[day]:
        for frame in annots[day][operation]:
            for pose in frame['poses']:
                gpdList.append(pose['gpdRep'])
                frameIds.append((day, operation, frame['frameIdx']))


gpdMatrix = np.array(gpdList)

In [ ]:
#checking shape, should be (1058,749)
np.shape(gpdMatrix)

## Reducing dimensionality


In [ ]:
#features in gpd are of different scales: dist in mm, angles between 0 and pi thus we standardize the values
scaler = StandardScaler()
gpdMatrixScaled = scaler.fit_transform(gpdMatrix)

In [ ]:
#fitting pca
pca = PCA()
pca.fit(gpdMatrixScaled)

In [ ]:
#plotting explained variance to find optimal number of components
cumVar = np.cumsum(pca.explained_variance_ratio_)
plt.plot(cumVar)
plt.xlabel('Number of components')
plt.ylabel('Cumulative explained variance')
plt.axhline(y=0.95, color='r', linestyle='--')
plt.show()


As shown in the plot, explained variability rises rapidly until 50 components then flattens significantly and crosses the 95% threshold at 100 components. Thus given that diversity features and scene graphs features have less components. We'll fit pca with 50 components so that it doesnt dominate as much.

In [ ]:
pca = PCA(n_components=50)
gpdReduced = pca.fit_transform(gpdMatrixScaled)

In [ ]:
#Averaging gpd per frame
frameGpd = defaultdict(list)

#appending corresponding poses to frame
for i, frameId in enumerate(frameIds):
    frameGpd[frameId].append(gpdReduced[i])

#averaging per frame
frameGpdMean = {}
for frameId, vectors in frameGpd.items():
    frameGpdMean[frameId] = np.mean(vectors, axis=0)

In [ ]:
#ensuring right number of frames (should be 626)
len(frameGpdMean)

In [ ]:
#adding gpd frame mean to annotations
for frameId, vector in frameGpdMean.items():
    for frame in annots[frameId[0]][frameId[1]]:
        if frameId[2] == frame['frameIdx']:
            frame['meanGpd'] = vector
            break

In [ ]:
#inspecting results
frame = annots['day1']['operation_1'][0]['meanGpd']
print(frame)


In [ ]:
print(np.shape(frame))

In [ ]:
##exporting annotations with added gpd
import pickle 

with open('gpdAnnots.pkl', 'wb') as f:
    pickle.dump(annots, f)
    